# MojoVec Python Quickstart

This notebook demonstrates the current managed Python API for MojoVec:

- creating Flat and SQ8 collections;
- adding vectors, metadata, and documents in batches;
- vector search with Chroma-style `where` filters;
- BM25 and hybrid search with RRF;
- `upsert`, soft deletion, statistics, and compaction;
- atomic persistence and memory-mapped loading.

MojoVec runs in-process. You do not need a server, manual memory allocation, or `free()` calls. Collection IDs are Python integers.

## 1. Installation

Install MojoVec from PyPI into the environment used by the current notebook kernel. Restart the kernel once after upgrading from an older version.

In [ ]:
%pip install mojovec

In [ ]:
import mojovec

## 2. Create a collection

Set `quantized=True` to use SQ8 storage. Set it to `False` for Flat Float32 storage; the rest of the API remains identical.

Supported distance metrics:

- `l2` — squared Euclidean distance;
- `cosine` — `1 - cosine_similarity`;
- `ip` — `1 - inner_product`.

For every metric, a smaller distance means a closer result.

In [ ]:
collection = mojovec.Collection(
    dimension=4,
    M=16,
    ef_construction=96,
    ef_search=64,
    quantized=True,
    name="knowledge_base",
    metric="cosine",
)

collection

## 3. Batch add: vectors + metadata + documents

The Python API accepts either nested vectors such as `[[...], [...]]` or a flattened row-major sequence. Metadata values can be `str`, `int`, `float`, or `bool`.

In [ ]:
collection.add(
    ids=[101, 202, 303, 404],
    embeddings=[
        [1.0, 0.0, 0.0, 0.0],
        [0.9, 0.1, 0.0, 0.0],
        [0.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0],
    ],
    metadatas=[
        {"category": "guide", "year": 2024, "published": True},
        {"category": "internals", "year": 2026, "published": True},
        {"category": "release", "year": 2026, "published": False},
        {"category": "tutorial", "year": 2027, "published": True},
    ],
    documents=[
        "Vector search introduction",
        "HNSW graph traversal and vector search",
        "Database release notes",
        "BM25 full text search tutorial",
    ],
)

print(collection.stats())
print(collection.get_metadata(202))
print(collection.get_document(202))

## 4. Vector search and `where` filters

`where` supports `$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`, `$in`, `$nin`, `$and`, `$or`, and `$not`.

Every managed query returns five keys: `ids`, `distances`, `metadatas`, `documents`, and `scores`. Vector search populates `distances`; BM25 and hybrid search populate `scores`.

In [ ]:
vector_result = collection.query(
    query_embeddings=[[1.0, 0.0, 0.0, 0.0]],
    n_results=3,
    where={
        "$and": [
            {"published": True},
            {"year": {"$gte": 2024}},
            {"category": {"$in": ["guide", "internals", "tutorial"]}},
        ]
    },
)

vector_result

In [ ]:
def show_results(result):
    for query_index, row_ids in enumerate(result["ids"]):
        print(f"query {query_index}")
        for rank, record_id in enumerate(row_ids):
            if record_id < 0:  # Padding when fewer than n_results matches exist.
                continue
            value = (
                result["distances"][query_index][rank]
                if result["distances"]
                else result["scores"][query_index][rank]
            )
            print(
                f"  rank={rank + 1} id={record_id} value={value:.6f}\n"
                f"    metadata={result['metadatas'][query_index][rank]}\n"
                f"    document={result['documents'][query_index][rank]}"
            )


show_results(vector_result)

## 5. BM25 document search

Passing `query_texts` selects BM25 search. The analyzer applies Unicode lowercase conversion, word boundaries, and built-in English and Russian stopwords without stemming.

In [ ]:
bm25_result = collection.query(
    query_texts=["HNSW vector search"],
    n_results=3,
    where={"published": True},
)

show_results(bm25_result)

## 6. Hybrid search with RRF

Hybrid search combines HNSW and BM25 rankings using reciprocal rank fusion. An embedding and text at the same batch position form one hybrid query.

In [ ]:
hybrid_result = collection.query_hybrid(
    query_embeddings=[[1.0, 0.0, 0.0, 0.0]],
    query_texts=["HNSW graph traversal"],
    n_results=3,
    rrf_k=60,
    candidate_multiplier=4,
    where={"published": True},
)

show_results(hybrid_result)

## 7. Update, upsert, delete, and compaction

- `add` accepts only new IDs;
- `upsert` inserts missing IDs and replaces existing records;
- `update` requires every ID to exist;
- `delete` performs soft deletion and ignores unknown IDs.

A vector-only update preserves existing metadata and documents. When a payload is provided explicitly, that payload is replaced in full.

In [ ]:
collection.upsert(
    ids=[202, 505],
    embeddings=[
        [0.85, 0.15, 0.0, 0.0],
        [0.0, 0.0, 0.9, 0.1],
    ],
    metadatas=[
        {"category": "internals", "year": 2028, "published": True},
        {"category": "guide", "year": 2028, "published": True},
    ],
    documents=[
        "Updated HNSW internals",
        "Persistence and mmap guide",
    ],
)
collection.delete([303])

print("before compaction:", collection.stats())
report = collection.compact_if_needed(deleted_ratio=0.20)
print("compaction report:", report)
print("after compaction:", collection.stats())

## 8. Atomic persistence and memory-mapped loading

`save()` publishes a checksummed snapshot through atomic file replacement. This small example uses `mmap_threshold_bytes=0` to force the memory-mapped loading path.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

temporary_directory = TemporaryDirectory(prefix="mojovec-notebook-")
database_path = Path(temporary_directory.name) / "knowledge_base.mojovec"

collection.save(database_path)
loaded = mojovec.load(
    database_path,
    memory_mapped=True,
    mmap_threshold_bytes=0,
)

print("memory mapped:", loaded.is_memory_mapped())
print("loaded stats:", loaded.stats())
show_results(loaded.query([[1.0, 0.0, 0.0, 0.0]], n_results=2))

## Next steps

- For bulk NumPy arrays, use `upsert_numpy()` and `query_numpy()` with contiguous `int64` IDs and `float32` embeddings.
- For crash recovery between snapshots, use `enable_wal()`, `flush_wal()`, `checkpoint()`, and `mojovec.recover()`.
- Tune `M`, `ef_construction`, and `ef_search` against recall and latency on your own embeddings.